# Review the deployment - predictions & live inference (read-only)

A data scientist's **read-only** view of what got promoted to production. As `ML_DEV_ROLE` we can *see* the prod model, predictions, and scoring task to QC the deployment - but we cannot change them (that's the pipeline's job).

Then we run the native-SQL `PREDICT_PROBA` model function on a few rows **in our dev sandbox** to show exactly how scoring works in production.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE ML_DEV_ROLE").collect()
session.sql("USE SECONDARY ROLES NONE").collect()   # operate purely as the data scientist
session.sql("USE WAREHOUSE CORTEX_CODE_WH").collect()

PROD  = "ML_FRAUD_PRODUCTION"
DEV   = "ML_FRAUD_DEV_SANDBOX"
MODEL = "AML_FRAUD_GBM"
print("role:", session.get_current_role())

## 1. Confirm what's live in production (read-only QC)

`ML_DEV_ROLE` has read-only visibility into prod - enough to verify a deployment landed, not to change it.

In [ ]:
# Confirm the promoted model, predictions, and scoring task are live in PROD.
print("== PROD model registry ==")
try:
    for r in session.sql(f"SHOW VERSIONS IN MODEL {PROD}.ML.{MODEL}").collect():
        d = r.as_dict()
        print(f"  {MODEL} {d['name']} | default={d['is_default_version']}")
except Exception as e:
    print("  (no prod model yet):", str(e).splitlines()[0][:80])

print("== PROD predictions ==")
try:
    n = session.sql(f"SELECT COUNT(*) AS C FROM {PROD}.ANALYTICS.PREDICTIONS").collect()[0]["C"]
    print(f"  PREDICTIONS rows: {n:,}")
except Exception as e:
    print("  (no predictions yet):", str(e).splitlines()[0][:80])

print("== PROD batch scoring task ==")
try:
    for r in session.sql(f"SHOW TASKS LIKE 'SCORE_BATCH_TASK' IN SCHEMA {PROD}.ANALYTICS").collect():
        d = r.as_dict()
        print(f"  {d['name']} | state={d['state']} | schedule={d.get('schedule')}")
except Exception as e:
    print("  (no task yet):", str(e).splitlines()[0][:80])

## 2. The production predictions

Read the batch output the downstream systems consume. The model should score known laundering far higher than normal activity.

In [ ]:
# Fraud vs normal: average score by true label (separation check).
session.sql(f"""
    SELECT CASE WHEN LABEL = 1 THEN 'fraud' ELSE 'normal' END AS CLASS,
           COUNT(*) AS N,
           ROUND(AVG(FRAUD_SCORE), 3) AS AVG_SCORE
    FROM {PROD}.ANALYTICS.PREDICTIONS
    GROUP BY 1 ORDER BY 1
""").to_pandas()

In [ ]:
# The highest-risk transactions the batch job flagged.
session.sql(f"""
    SELECT ACCOUNT_ID, EVENT_TS, PAYMENT_FORMAT, LABEL,
           ROUND(FRAUD_SCORE, 4) AS FRAUD_SCORE
    FROM {PROD}.ANALYTICS.PREDICTIONS
    ORDER BY FRAUD_SCORE DESC
    LIMIT 10
""").to_pandas()

## 3. How inference runs - a live sample scored in dev

The scoring is a **native SQL** call to the model: `<model>!PREDICT_PROBA(features...)`. We review prod read-only, but we can run this live in our **own dev sandbox** on a few rows to see exactly what the batch task does in production.

In [ ]:
# Build the native-SQL PREDICT_PROBA call from the dev model's signature, then run it
# on a few TEST rows in dev. (Same mechanics the prod batch task uses, just scaled down.)
from snowflake.ml.registry import Registry

mv  = Registry(session=session, database_name=DEV, schema_name="ML").get_model(MODEL).default
fn  = next(f for f in mv.show_functions() if f["target_method"].lower() == "predict_proba")
sig = [str(s.name).upper() for s in fn["signature"].inputs]

REQUEST_CTX = {"AMOUNT_PAID","IS_CROSS_CURRENCY","IS_CROSS_BORDER","IS_HIGH_RISK_FORMAT","AMOUNT_TO_AVG_RATIO"}
PROFILE_HIST = {"HIST_TXN_COUNT","HIST_AVG_AMOUNT","HIST_STD_AMOUNT","HIST_MAX_AMOUNT","HIST_DISTINCT_RECEIVERS",
                "HIST_DISTINCT_RECEIVER_BANKS","HIST_DISTINCT_COUNTRIES","HIST_FOREIGN_CCY_SHARE","HIST_HIGH_RISK_SHARE"}

def expr(n):
    if n.startswith("PAYMENT_FORMAT_"):
        return f"IFF(UPPER(REPLACE(s.PAYMENT_FORMAT,' ','_'))='{n[len('PAYMENT_FORMAT_'):]}',1,0)"
    if n in REQUEST_CTX:            return f"COALESCE(s.{n},0)"
    if n == "HIST_AMOUNT_CV":       return "COALESCE(h.HIST_STD_AMOUNT/NULLIF(h.HIST_AVG_AMOUNT,0),0)"
    if n == "HIST_RECEIVER_FANOUT": return "COALESCE(h.HIST_DISTINCT_RECEIVERS/NULLIF(h.HIST_TXN_COUNT,0),0)"
    if n in PROFILE_HIST:           return f"COALESCE(h.{n},0)"
    return "0"

args = ", ".join(expr(n) for n in sig)
model_fqn = f"{DEV}.ML.{MODEL}"
sql = f"""
WITH scored AS (
    SELECT s.ACCOUNT_ID, s.PAYMENT_FORMAT, s.IS_LAUNDERING AS LABEL,
           {model_fqn}!PREDICT_PROBA({args}) AS PRED
    FROM {DEV}.CURATED.FRAUD_ABT s
    LEFT JOIN {PROD}.CURATED.ACCOUNT_HISTORY h ON s.ACCOUNT_ID = h.ACCOUNT_ID
    WHERE s.SPLIT = 'TEST'
    LIMIT 500
)
SELECT ACCOUNT_ID, PAYMENT_FORMAT, LABEL,
       ROUND(PRED:output_feature_1::FLOAT, 4) AS FRAUD_SCORE
FROM scored ORDER BY FRAUD_SCORE DESC LIMIT 10
"""
print(sql)                     # show the native-SQL PREDICT_PROBA call
session.sql(sql).to_pandas()   # run it live in dev

## In production, this runs on a schedule

The exact same native-SQL `PREDICT_PROBA` call is what **`SCORE_BATCH_TASK`** runs daily in prod - scoring the latest events into `PREDICTIONS` with no human in the loop. Here we ran it read-only in dev on a few rows to see the mechanics; the task does it at scale, on schedule, as the service account.